In [1]:
#imports 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report

In [2]:
tdf=pd.read_csv("data_center_hybrid.csv")
print(tdf.index)
print("All columns are")
ai=0
for i in tdf:
    print(ai,i)
    ai+=1

RangeIndex(start=0, stop=126770, step=1)
All columns are
0 Year
1 Facility_ID
2 Facility_Name
3 Owner_Company
4 City
5 Country
6 Facility_Type
7 Estimated_Capacity_MW
8 PUE
9 Cooling_System_Type
10 WUE_L_per_kWh
11 Daily_Electricity_Usage_MWh
12 Daily_Water_Usage_Gallons
13 Surrounding_Water_Stress_Tier


In [3]:
tdf.head(n=10)

,Year,Facility_ID,Facility_Name,Owner_Company,City,Country,Facility_Type,Estimated_Capacity_MW,PUE,Cooling_System_Type,WUE_L_per_kWh,Daily_Electricity_Usage_MWh,Daily_Water_Usage_Gallons,Surrounding_Water_Stress_Tier
0,2019,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.24,1.975,Evaporative,1.481,183.62,36362.94,Low
1,2020,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.36,1.967,Evaporative,1.459,254.34,49833.60,Low
2,2021,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.47,1.928,Evaporative,1.450,266.85,53026.35,Low
3,2022,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.59,1.897,Evaporative,1.413,199.14,39198.30,Low
4,2023,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.70,1.869,Evaporative,1.389,225.94,44366.48,Low
5,2024,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.82,1.842,Evaporative,1.369,195.72,38435.64,Low
6,2025,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.95,1.815,Evaporative,1.341,229.83,44852.92,Low
7,2019,DC-97AB1D99,Central Office 2,StarHub Ltd.,Singapore,Singapore,Enterprise/Standard,8.20,1.796,Air Cooled,0.093,329.47,4488.46,Low
8,2020,DC-97AB1D99,Central Office 2,StarHub Ltd.,Singapore,Singapore,Enterprise/Standard,8.35,1.792,Air Cooled,0.092,253.65,3430.72,Low
9,2021,DC-97AB1D99,Central Office 2,StarHub Ltd.,Singapore,Singapore,Enterprise/Standard,8.50,1.759,Air Cooled,0.091,270.58,3679.70,Low


In [4]:
tdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126770 entries, 0 to 126769
Data columns (total 14 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Year                           126770 non-null  int64  
 1   Facility_ID                    126770 non-null  object 
 2   Facility_Name                  126770 non-null  object 
 3   Owner_Company                  126770 non-null  object 
 4   City                           126770 non-null  object 
 5   Country                        126770 non-null  object 
 6   Facility_Type                  126770 non-null  object 
 7   Estimated_Capacity_MW          126770 non-null  float64
 8   PUE                            126770 non-null  float64
 9   Cooling_System_Type            126770 non-null  object 
 10  WUE_L_per_kWh                  126770 non-null  float64
 11  Daily_Electricity_Usage_MWh    126770 non-null  float64
 12  Daily_Water_Usage_Gallons     

In [5]:
for col in tdf.select_dtypes(include=['object', 'category']):
    print(tdf[col].value_counts())
    print("-" * 20)  # Separator line


Facility_ID
DC-D2763E00    7
DC-21C0EA0D    7
DC-518B7F4C    7
DC-6D0E9079    7
DC-B0E5037A    7
              ..
DC-1C259F9E    7
DC-3CAF05F5    7
DC-232F02F1    7
DC-B2E3F8DD    7
DC-31D8B15E    7
Name: count, Length: 18110, dtype: int64
--------------------
Facility_Name
Amsterdam                     231
Singapore                     217
London                        189
Sydney                        161
Frankfurt                     154
                             ... 
Northland Utica Datacenter      7
SS&C GlobeOp Data Center        7
Hivelocity - Seattle 1          7
2020 5th Avenue (SEA11)         7
ICS_DataCenter_216              7
Name: count, Length: 16286, dtype: int64
--------------------
Owner_Company
Amazon AWS                                               3731
Equinix                                                  3374
Lumen                                                    2821
China Telecom                                            2212
Zenlayer                   

In [6]:
tdf.columns

Index(['Year', 'Facility_ID', 'Facility_Name', 'Owner_Company', 'City',
       'Country', 'Facility_Type', 'Estimated_Capacity_MW', 'PUE',
       'Cooling_System_Type', 'WUE_L_per_kWh', 'Daily_Electricity_Usage_MWh',
       'Daily_Water_Usage_Gallons', 'Surrounding_Water_Stress_Tier'],
      dtype='object')

In [7]:
# Data Cleaning
df = tdf
df = df.drop(columns=['Facility_ID', 'Facility_Name'])

In [8]:
df.columns

Index(['Year', 'Owner_Company', 'City', 'Country', 'Facility_Type',
       'Estimated_Capacity_MW', 'PUE', 'Cooling_System_Type', 'WUE_L_per_kWh',
       'Daily_Electricity_Usage_MWh', 'Daily_Water_Usage_Gallons',
       'Surrounding_Water_Stress_Tier'],
      dtype='object')

In [9]:
# Cat Boost

X = df.drop("Surrounding_Water_Stress_Tier", axis=1)
y = df["Surrounding_Water_Stress_Tier"]

Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


cat_features = X.select_dtypes(include='object').columns.tolist()

cb = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.1,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    random_state=42,
    verbose=0
)

cb.fit(
    Xtrain,
    ytrain,
    cat_features=cat_features
)

ypred = cb.predict(Xtest)

print("Test Accuracy:", accuracy_score(ytest, ypred))
print()
print(classification_report(ytest, ypred))


Test Accuracy: 0.8277983750098604

              precision    recall  f1-score   support

        High       0.80      0.86      0.83      9047
         Low       0.89      0.77      0.82      6964
      Medium       0.82      0.84      0.83      9343

    accuracy                           0.83     25354
   macro avg       0.84      0.82      0.83     25354
weighted avg       0.83      0.83      0.83     25354



In [10]:
cv_scores = cross_val_score(
    cb,
    X,
    y,
    cv=5,
    scoring='accuracy',
    params={'cat_features': cat_features}
)

print("CV Accuracy Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

CV Accuracy Scores: [0.35868108 0.33375404 0.36195472 0.27613    0.3165181 ]
Mean CV Accuracy: 0.329407588546186
